# E17 — Robustness, consistency & gap-closure consolidation

**Spec:** `EXPERIMENT_PLAN.md` §E17 (revised 2026-09-21). Three parts:

- **H1 — robustness** *(scoped to **persistence, GBM and GRU**; MC-dropout / E8 is excluded —
  see §2a)*. Are the headline results stable under reasonable alternative statistical choices? (i) the bootstrap's clustering assumption (Q-STAT-03c), via a
  **mission-level cluster bootstrap**; (ii) the weight-clipping cap (Q-SEL-03) — a
  *confirmation that clipping stays untriggered under stricter triggers*, not a live sweep,
  because k̂ never approached the trigger; (iii) the declared multiple-comparison policy
  (Q-STAT-04), applied to the family of p-values E11 already recorded.
- **H2 — gap closure.** The **one-sided CQR** coverage cell, completing the
  {split, CQR} × {two-sided, one-sided} restoration matrix. This is the only genuinely new
  scientific quantity in E17.
- **H3 — synthesis.** Is there a shared *per-event* mechanism behind the five manifestations
  of the train/test high-risk imbalance? An honest null is a valid outcome and is reported as
  one.

> **No new formal test is introduced** (Q-STAT-04's single-primary-contrast policy). H2 inherits
> E11/E12's coverage convention exactly so its cell is directly comparable to the other three;
> H3 is descriptive association only.

In [ ]:
# --- Setup + provenance (invariant I4) ------------------------------------------------------
import json, subprocess, sys
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal.models import robustness_runner as RR
from kelvins_conformal.reporting import write_table_atomic

cfg = load_config()
PREFIX = "e17_"
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT),
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "UNAVAILABLE"

def save_table(df, name):
    write_table_atomic(df, TABDIR / f"{PREFIX}{name}.csv"); print(f"saved: reports/tables/{PREFIX}{name}.csv")

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{PREFIX}{name}.{ext}", dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
    print(f"saved: reports/figures/{PREFIX}{name}.png|pdf")

PROVENANCE = {"experiment_ids": ["E17"], "analysis": "robustness, consistency & gap-closure consolidation",
              "git_commit_sha": git_sha(), "config_hash": cfg.config_hash,
              "executed_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0]}
print(json.dumps(PROVENANCE, indent=2))

SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, GREEN, RED = "#2a78d6", "#eb6834", "#1baf7a", "#d6453a"
plt.rcParams.update({"axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": MUTED,
                     "ytick.color": MUTED, "text.color": INK, "axes.titlecolor": INK})

## 1. Run

In [ ]:
RES = RR.run_e17(cfg)
meta = RES["meta"]
print(json.dumps(meta, indent=2, default=str))
for key in ("h1_cluster_bootstrap", "h1_gate2_verdict", "h1_clipping", "h1_multiple_comparison",
            "h2_coverage", "h2_per_seed", "coverage_restoration_matrix", "h3_association", "h3_overlap"):
    save_table(RES[key], key)
PRIM = meta["primary_level"]

## 2a. Scope of H1 — which learners are checked, and which is not

**MC-dropout (the E8 Bayesian arm) is NOT included in H1's robustness checks.** This is a
disclosed scope boundary, decided by Sidh on 2026-09-21 before E17 was re-run, not an oversight
and not a silent gap.

**Why.** No `e8_mcdropout` hyperparameter cache exists under the current config hash. E8 published
under an *older* hash — the configuration has since gained the `decision_cost` and
`threshold_analysis` blocks — so running a fresh search now would select hyperparameters **E8
never used**. E17's Bayesian arm would then be a different fitted model from the one E8 reported,
and every cross-reference between this robustness supplement and E8's findings would compare two
models while appearing to compare one.

**What this does and does not limit.** The Gate-2 headline contrast — the project's only formally
tested result — is **persistence**, which has no hyperparameters and needs no search; it is inside
the retained scope. **H2 and H3 are unaffected**: H2 needs only the GBM quantile heads and H3 the
GBM point residual. **E8's own published results are unchanged by this**; the decision governs
only which arms E17's robustness checks cover.

In [ ]:
display(RES["h1_excluded_learners"])
print("H1 covers:", meta["h1"]["h1_learners"])
print("H1 excludes:", meta["h1"]["h1_excluded_learners"])
print()
print("Reason on record:")
print(" ", meta["h1"]["h1_exclusion_reason"])

## 2. H1 (i) — clustering assumption: mission-level cluster bootstrap vs the event bootstrap (Q-STAT-03c)

The **same** per-event covered indicators feed both schemes; only the resampling differs. A
`width_ratio` above 1 means the cluster bootstrap is wider — the expected direction when events
within a mission are positively correlated.

In [ ]:
cb = RES["h1_cluster_bootstrap"]
print(f"clusters: {meta['h1']['n_clusters']} missions over {meta['h1']['n_supported_events']} supported events; "
      f"largest cluster holds {100*cb['largest_cluster_frac'].iloc[0]:.1f}% of events")
head = cb[(cb["nominal"] == PRIM) & (cb["sided"] == "two")].copy()
show = pd.DataFrame({
    "learner": head["learner"], "method": head["method"], "coverage": head["coverage"].round(4),
    "iid 95% CI": [f"[{r.iid_lo:.4f}, {r.iid_hi:.4f}]" for r in head.itertuples()],
    "cluster 95% CI": [f"[{r.cluster_lo:.4f}, {r.cluster_hi:.4f}]" for r in head.itertuples()],
    "width ratio (cluster/iid)": head["width_ratio"].round(2),
}).reset_index(drop=True)
display(show)
print(f"\nwidth ratio across ALL arms and levels: median {cb['width_ratio'].median():.2f}, "
      f"range [{cb['width_ratio'].min():.2f}, {cb['width_ratio'].max():.2f}]")

### Does the Gate-2 conclusion survive the clustering assumption?

The Gate-2 claim is that rule-weighting **restores** two-sided marginal coverage: nominal should
sit inside the weighted arm's interval and outside the naive arm's. Checked under both schemes.

In [ ]:
v = RES["h1_gate2_verdict"]
vp = v[v["nominal"] == PRIM].copy()
vp["restored"] = vp["weighted_covers_nominal"] & ~vp["naive_covers_nominal"]
display(vp[["scheme", "learner", "naive_coverage", "weighted_coverage",
            "naive_covers_nominal", "weighted_covers_nominal", "restored"]].round(4).reset_index(drop=True))
agree = vp.pivot_table(index="learner", columns="scheme", values="restored").astype(bool)
print("\nSame verdict under both schemes for every learner:",
      bool((agree["iid"] == agree["cluster"]).all()))
print("Learners where weighting restores coverage (iid):", sorted(agree.index[agree["iid"]]))
print("Learners where weighting restores coverage (cluster):", sorted(agree.index[agree["cluster"]]))

## 3. H1 (ii) — weight-clipping cap (Q-SEL-03)

Q-SEL-03 Decision B makes clipping **conditional**: it fires only when Pareto k̂ exceeds the
trigger. It never fired. This confirms it still does not fire at progressively stricter triggers,
and reports the induced bias Δ_B that clipping *would* introduce (0 when untriggered).

In [ ]:
clip = RES["h1_clipping"]
display(clip[["weights", "khat", "khat_band", "khat_trigger", "clipping_triggered",
              "clipped_fraction", "bias_delta", "n_calibration", "n_effective"]].round(4).reset_index(drop=True))
print("\nClipping triggered anywhere:", bool(clip["clipping_triggered"].any()))
for w in clip["weights"].unique():
    sub = clip[clip["weights"] == w]
    k, ne, n = sub["khat"].iloc[0], sub["n_effective"].iloc[0], sub["n_calibration"].iloc[0]
    print(f"  {w}: k-hat = {k:.3f} ({sub['khat_band'].iloc[0]}); "
          f"effective n = {ne:.1f} of {n} ({100*ne/n:.1f}%)")

## 4. H1 (iii) — multiple-comparison policy (Q-STAT-04)

Holm–Bonferroni applied to the family of p-values E11 **already recorded**. No new test is run;
this only confirms the pre-registered primary contrast survives the declared policy.

In [ ]:
mc = RES["h1_multiple_comparison"]
pcol = next(c for c in mc.columns if c.lower() in ("p_value", "p", "pvalue"))
cols = [c for c in ("learner", "method", "sided", "nominal") if c in mc.columns]
display(mc[cols + [pcol, "p_adjusted_holm", "rejected_holm", "n_tests_in_family"]].reset_index(drop=True))
print(f"\nfamily size {mc['n_tests_in_family'].iloc[0]}; rejected at alpha=0.05 after Holm: "
      f"{int(mc['rejected_holm'].sum())} of {len(mc)}")
print(f"smallest raw p = {mc[pcol].min():.3g}; its Holm-adjusted p = "
      f"{mc.loc[mc[pcol].idxmin(), 'p_adjusted_holm']:.3g}; "
      f"still rejected = {bool(mc.loc[mc[pcol].idxmin(), 'rejected_holm'])}")

## 5. H2 — the completed coverage-restoration matrix (the new cell)

Three cells are reloaded from the committed E9/E11 and E12 tables — exactly the numbers those
experiments published. The fourth, **CQR one-sided**, is computed here for the first time.

"RESTORED" requires the rule-weighted arm's interval to contain nominal **and** the naive arm's
not to. If both contain nominal there was no deficit to restore, which is a different statement
and is labelled as such.

In [ ]:
mx = RES["coverage_restoration_matrix"]
display(mx[["family", "sided", "source", "naive_coverage", "naive_ci", "weighted_coverage",
            "weighted_ci", "change_pp", "verdict"]].round(4).reset_index(drop=True))
print("\nH2 cell (CQR, one-sided), all levels:")
h2 = RES["h2_coverage"]
display(h2[["method", "nominal", "coverage_mean", "coverage_sd", "n", "cp_lo_mean", "cp_hi_mean",
            "gap_pp", "frac_inf_width"]].round(4).reset_index(drop=True))

In [ ]:
# The 2x2 matrix as a figure: the clearest single statement of where the correction works.
fig, ax = plt.subplots(figsize=(7.2, 4.2), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
fams, sides = ["split conformal", "CQR"], ["two", "upper"]
label = {"two": "two-sided", "upper": "one-sided (upper)"}
for i, fam in enumerate(fams):
    for j, sd in enumerate(sides):
        r = mx[(mx["family"] == fam) & (mx["sided"] == sd)].iloc[0]
        restored = r["verdict"] == "RESTORED"
        colour = GREEN if restored else (MUTED if r["verdict"] == "no deficit to restore" else RED)
        ax.add_patch(plt.Rectangle((j, -i), 1, 1, facecolor=colour, alpha=0.16, edgecolor=AXIS, lw=1.2))
        ax.text(j + 0.5, -i + 0.72, r["verdict"], ha="center", va="center", fontsize=11,
                color="#0a7a52" if restored else colour, fontweight="bold")
        ax.text(j + 0.5, -i + 0.46, f"naive {r['naive_coverage']:.3f} -> weighted {r['weighted_coverage']:.3f}",
                ha="center", va="center", fontsize=9, color=INK)
        ax.text(j + 0.5, -i + 0.26, f"{r['change_pp']:+.1f} pp   ({r['source']})",
                ha="center", va="center", fontsize=8, color=INK2)
ax.set_xlim(0, 2); ax.set_ylim(-1, 1)
ax.set_xticks([0.5, 1.5]); ax.set_xticklabels([label[s] for s in sides], fontsize=10)
ax.set_yticks([0.5, -0.5]); ax.set_yticklabels(fams, fontsize=10)
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title(f"Does rule-derived weighting restore coverage? Official test set, nominal {PRIM:.0%}",
             loc="left", fontsize=11)
fig.text(0.0, -0.04, "Each cell: naive -> rule-weighted empirical coverage, and the change in "
         "percentage points. 'RESTORED' means the weighted interval contains nominal and the naive "
         "one does not.", fontsize=7.5, color=INK2, ha="left", va="top", wrap=True)
fig.tight_layout()
save_fig(fig, "coverage_restoration_matrix")
plt.close(fig)

## 6. H3 — is there one shared per-event mechanism behind the five manifestations?

**Stated before looking:** only three of the five manifestations are event-level quantities at
all. M4 (grid-instrument distortion) and M5 (the asymmetric partial fix) are properties of the
threshold *grid* and of a selection criterion over it — they have no per-event membership. A
per-event diagnostic therefore **cannot**, even in principle, unify all five. The most it can do
is unify M1–M3, and that is the question actually being asked here.

In [ ]:
assoc = RES["h3_association"]
display(assoc[["manifestation", "description", "level_declared", "n_members",
               "diagnostic_mean_in", "diagnostic_mean_out", "high_risk_share_in",
               "high_risk_share_out", "point_biserial_r"]].round(4))
ov = RES["h3_overlap"]
display(ov.round(3))
ev = assoc[assoc["level_declared"] == "event"]
R_MIN = float(ev["point_biserial_r"].abs().min())
LIFT_MIN = float(ov["lift"].min())
print(f"\nDiagnostic: {meta['h3']['diagnostic']} over {meta['h3']['n_events']} supported events.")
print(f"Weakest |point-biserial r| across M1-M3: {R_MIN:.3f}; weakest pairwise lift: {LIFT_MIN:.2f}x")

## 7. Summary — measurement only

In [ ]:
restored = mx[mx["verdict"] == "RESTORED"]
not_restored = mx[mx["verdict"] == "not restored"]
_p = RES["h1_gate2_verdict"].pivot_table(index=["learner", "nominal"], columns="scheme",
                                         values="weighted_covers_nominal").astype(bool)
same_verdict = bool((_p["iid"] == _p["cluster"]).all())
cqr_up = mx[(mx.family == "CQR") & (mx.sided == "upper")].iloc[0]
print(f"""
E17 - ROBUSTNESS, CONSISTENCY & GAP-CLOSURE (measurement only, exactly as observed)

 H1 SCOPE: {meta['h1']['h1_learners']} checked; {meta['h1']['h1_excluded_learners']} EXCLUDED (stale E8
   hyperparameter cache under the current config hash - disclosed, see §2a).
 H1 (i) clustering: {meta['h1']['n_clusters']} missions over {meta['h1']['n_supported_events']} supported events.
   Cluster-vs-iid CI width ratio: median {cb['width_ratio'].median():.2f}, range [{cb['width_ratio'].min():.2f}, {cb['width_ratio'].max():.2f}].
   Gate-2 verdict identical under both resampling schemes: {same_verdict}.
 H1 (ii) clipping: triggered anywhere across triggers 0.7 / 0.5 / 0.3: {bool(clip['clipping_triggered'].any())}.
 H1 (iii) Holm over the recorded family of {mc['n_tests_in_family'].iloc[0]}: primary contrast still rejected =
   {bool(mc.loc[mc[pcol].idxmin(), 'rejected_holm'])} (adjusted p = {mc.loc[mc[pcol].idxmin(), 'p_adjusted_holm']:.3g}).
 H2 coverage-restoration matrix, nominal {PRIM:.0%}: RESTORED in {len(restored)} of 4 cells
   ({', '.join(f"{r.family}/{r.sided}" for r in restored.itertuples()) or 'none'});
   not restored in {len(not_restored)} ({', '.join(f"{r.family}/{r.sided}" for r in not_restored.itertuples()) or 'none'}).
   The NEW cell (CQR, one-sided): naive {cqr_up.naive_coverage:.4f} -> weighted {cqr_up.weighted_coverage:.4f}
   ({cqr_up.change_pp:+.1f} pp), verdict {cqr_up.verdict}.
 H3: {meta['h3']['event_level_manifestations']} of 5 manifestations are event-level; M4/M5 are instrument-level
   and have no per-event membership to predict. Weakest |r| across M1-M3 = {R_MIN:.3f}; weakest pairwise lift = {LIFT_MIN:.2f}x.

 NOT DECIDED HERE: whether the completed matrix changes any manuscript claim; how the H3 outcome
 is framed; anything in E18.
""")
(cfg.path("reports_dir") / "06_robustness_provenance.json").write_text(
    json.dumps({**PROVENANCE, "timings": meta["timings"]}, indent=2, default=str), encoding="utf-8")
print("provenance:", cfg.path("reports_dir") / "06_robustness_provenance.json")